# freqgen Colab v2 — Correct SPAI Setup (matches paper's original env)

**Key lesson from v1:** SPAI requires `numpy~=1.26.4`. Never upgrade numpy.
Install SPAI requirements FIRST, then do everything else in that environment.

**Inference command (from README):** `python -m spai infer --input <dir> --output <dir>`
No `--cfg` needed — defaults to `./weights/spai.pth` and `./configs/spai.yaml`.

Runtime: `Runtime → Change runtime type → T4 GPU`

## 1. Install SPAI (numpy 1.26.4, matches paper)

In [ ]:
import subprocess, sys

# Step 1: clone SPAI
import os
if not os.path.exists('/content/spai'):
    !git clone https://github.com/mever-team/spai.git /content/spai -q
%cd /content/spai

# Step 2: install PyTorch first (conda-style, via pip with CUDA)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Step 3: install SPAI requirements EXACTLY as specified (numpy 1.26.4)
# Do NOT upgrade numpy — SPAI requires ~=1.26.4
!pip install -r requirements.txt filetype -q

import numpy as np
print(f'numpy {np.__version__}  (should be 1.26.x)')
import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print('SPAI install OK')

## 2. Download SPAI weights

In [ ]:
import os
os.makedirs('/content/spai/weights', exist_ok=True)
if not os.path.exists('/content/spai/weights/spai.pth'):
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /content/spai/weights/spai.pth
sz = os.path.getsize('/content/spai/weights/spai.pth') // 1_000_000
print(f'Weights: {sz} MB  (expect ~934)')

## 3. Download data

COCO val2017 for real images (~778 MB) + Synthbuster SD1.4 fakes (~12 GB).
Synthbuster download takes ~15 min — the cell checks if already present.

In [ ]:
import os, glob
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

# COCO val2017 (real images)
COCO = f'{DATA}/coco_val2017'
if not os.path.exists(COCO) or len(os.listdir(COCO)) < 100:
    print('Downloading COCO val2017 (~778 MB)...')
    !wget -q -c 'http://images.cocodataset.org/zips/val2017.zip' -O /tmp/cv.zip
    !unzip -q /tmp/cv.zip -d {DATA}
    os.rename(f'{DATA}/val2017', COCO)
    os.remove('/tmp/cv.zip')
print(f'Real: {len(os.listdir(COCO))} COCO images')

# Synthbuster SD1.4 fakes
SYNTH = f'{DATA}/synthbuster/stable-diffusion-1-4'
if not os.path.exists(SYNTH) or len(os.listdir(SYNTH)) < 100:
    print('Downloading Synthbuster (~12 GB, ~15 min)...')
    !wget -L -c 'https://zenodo.org/records/10066460/files/synthbuster.zip' -O /tmp/sb.zip
    sz = os.path.getsize('/tmp/sb.zip') // 1_000_000
    print(f'Downloaded: {sz} MB')
    !unzip -q /tmp/sb.zip 'synthbuster/stable-diffusion-1-4/*' -d {DATA}
    os.remove('/tmp/sb.zip')
fake_n = len(glob.glob(f'{SYNTH}/*.png')) if os.path.exists(SYNTH) else 0
print(f'Fake: {fake_n} Synthbuster SD1.4 images')

## 4. Spectral matching attack

Run the freqgen attack: rewrite each fake's Fourier magnitude to match the
real spectral target. Phase (structure/content) is preserved.

In [ ]:
import numpy as np, glob, os
from PIL import Image
from tqdm.notebook import tqdm

SIZE = 256
DATA = '/content/data'

def load_gray(p):
    return np.array(Image.open(p).convert('L').resize((SIZE,SIZE)), dtype=np.float64)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel())
    c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def spectral_match(img, target, gain_clip=(0.1, 12.0)):
    F = np.fft.fftshift(np.fft.fft2(img))
    cy, cx = img.shape[0]//2, img.shape[1]//2
    y, x = np.ogrid[:img.shape[0], :img.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = len(target)
    gain = np.clip(target / (radial_profile(img) + 1e-12), *gain_clip)
    gain[0] = 1.0  # preserve DC / brightness
    gmap = gain[np.clip(r, 0, mr-1)]
    gmap[r >= mr] = 1.0; gmap[r == 0] = 1.0
    return np.clip(np.fft.ifft2(np.fft.ifftshift(F * gmap)).real, 0, 255)

real_paths  = sorted(glob.glob(f'{DATA}/coco_val2017/*.jpg'))[:100]
fake_paths  = sorted(glob.glob(f'{DATA}/synthbuster/stable-diffusion-1-4/*.png'))[:100]
print(f'real: {len(real_paths)}  fake: {len(fake_paths)}')

# Build real spectral target
print('Building real spectral target...')
target = np.mean([radial_profile(load_gray(p)) for p in tqdm(real_paths[:50])], axis=0)

# Run attack
MATCHED = f'{DATA}/matched_sd14'
os.makedirs(MATCHED, exist_ok=True)
matched_paths = []
print('Running spectral matching attack...')
for p in tqdm(fake_paths[:60]):
    m = spectral_match(load_gray(p), target)
    out = f'{MATCHED}/{os.path.basename(p)}'
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)
print(f'Saved {len(matched_paths)} matched fakes')

# Measure the gap
N = 30
rh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in real_paths[:N]])
fh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in fake_paths[:N]])
mh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in matched_paths[:N]])
print(f'\n=== SPECTRAL GAP ===')
print(f'High-band  real={rh:.1f}  fake={fh:.1f}  matched={mh:.1f}')
print(f'Gap real/fake={rh/fh:.2f}x    real/matched={rh/mh:.2f}x')

## 5. Prepare image directories for SPAI

SPAI inference takes a **directory** of images (from README). Copy/symlink
real, fake, and matched images into separate input folders.

In [ ]:
import os, shutil, glob
DATA = '/content/data'

SPAI_IN = '/content/spai_input'
for tag, paths in [
    ('real',    sorted(glob.glob(f'{DATA}/coco_val2017/*.jpg'))[:50]),
    ('fake',    sorted(glob.glob(f'{DATA}/synthbuster/stable-diffusion-1-4/*.png'))[:50]),
    ('matched', sorted(glob.glob(f'{DATA}/matched_sd14/*.png'))[:50]),
]:
    d = f'{SPAI_IN}/{tag}'
    os.makedirs(d, exist_ok=True)
    for p in paths:
        dst = f'{d}/{os.path.basename(p)}'
        if not os.path.exists(dst):
            shutil.copy2(p, dst)
    print(f'{tag}: {len(os.listdir(d))} images in {d}')

## 6. SPAI inference — does the attack fool the SOTA detector?

Using the exact command from the README. Working dir must be `/content/spai`
so that `./weights/spai.pth` and `./configs/spai.yaml` resolve correctly.

In [ ]:
import os
os.makedirs('/content/spai_output/real',    exist_ok=True)
os.makedirs('/content/spai_output/fake',    exist_ok=True)
os.makedirs('/content/spai_output/matched', exist_ok=True)

%cd /content/spai

print('Running SPAI on REAL images...')
!python -m spai infer \
    --input /content/spai_input/real \
    --output /content/spai_output/real

print('Running SPAI on FAKE images...')
!python -m spai infer \
    --input /content/spai_input/fake \
    --output /content/spai_output/fake

print('Running SPAI on MATCHED (attacked) fakes...')
!python -m spai infer \
    --input /content/spai_input/matched \
    --output /content/spai_output/matched

print('SPAI inference done')

## 7. Results — evasion table

In [ ]:
import pandas as pd, glob, numpy as np

def load_scores(tag):
    csvs = glob.glob(f'/content/spai_output/{tag}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError(f'No SPAI output for {tag}. Check cell 6.')
    df = pd.read_csv(csvs[0])
    print(f'{tag}: {len(df)} rows, columns: {list(df.columns)}')
    return df

res_real    = load_scores('real')
res_fake    = load_scores('fake')
res_matched = load_scores('matched')

# SPAI outputs a score column — find it
score_col = [c for c in res_fake.columns if 'score' in c.lower() or 'pred' in c.lower()][0]
print(f'Score column: {score_col}')

fake_det    = (res_fake[score_col]    >= 0.5).mean()
matched_det = (res_matched[score_col] >= 0.5).mean()
real_fp     = (res_real[score_col]    >= 0.5).mean()

print()
print('='*55)
print('freqgen — SPAI Evasion Table')
print('='*55)
print(f'SPAI detects raw fakes:     {fake_det:.0%}')
print(f'SPAI detects matched fakes: {matched_det:.0%}')
print(f'SPAI false-positives real:  {real_fp:.0%}')
print(f'Evasion rate (attack):      {1-matched_det:.0%}')
print('='*55)
if 1-matched_det > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap in CVPR 2025 SOTA')
else:
    print('RESULT: SPAI survives attack -> learned detectors are robust')